# Сначала география, затем текст: кортеж и сумма скоров

Ближайшие N объявлений → скоры пяти моделей для **каждого** кандидата → итоговые 50.
Проверяем N = 50, 500, 5000, тот же split и выборку запросов, что в Hybrid_geo_pipeline.
Глобальные текстовые top-N не используются: релевантное объявление может быть далеко
в текстовой выдаче, но попасть в геопул. Dense берёт векторы items из готового FAISS;
BM25 использует статистики общего корпуса, без переобучения на геопуле.

Два сценария:
1. Кортеж текстовых скоров в порядке SCORE_ORDER, каждый по убыванию.
   Только при полном равенстве кортежа сравниваем расстояние, затем item_id.
   Первый скор определяет порядок; остальные разрешают ничьи, поэтому приоритет важен.
2. Сумма min–max нормированных скоров внутри top-N одного запроса. Диапазон каждого
   источника — [0, 1]; по умолчанию веса равны. Константный или отсутствующий сигнал даёт 0.
   При равенстве суммы сравниваем расстояние, затем item_id. Расстояние в сумму не входит.

Сохраняем скоры для максимального геопула, затем меняем N и способ сортировки без
повторного запуска моделей. Min–max пересчитывается для каждого N. Такая нормировка
выравнивает диапазоны, но не калибрует вероятность релевантности — проверяем её по Recall.

In [ ]:
from pathlib import Path

REPO_PATH = None  # путь к обновлённому репозиторию; None — поиск рядом с ноутбуком
DATA_PATH = None  # train.parquet; None — repo/data или /kaggle/input
DENSE_INDEX_DIR = None  # папка с giga/ и bge/; локально repo/dense_indexes, Kaggle /kaggle/working/dense_indexes
INSTALL_DEPENDENCIES = False  # True для нового окружения Kaggle
DENSE_MODELS = ["giga", "bge"]  # [] — быстрый эксперимент только с BM25
DEVICE = None  # автоматический выбор CPU/CUDA

SEED = 42
N_QUERIES = 500  # на каждую из val/test; None — все
NS = [50, 500, 5000]
K = 50
QUERY_BATCH_SIZE = 16  # ограничивает память при поиске и объединении
DENSE_BATCH_SIZE = 16
BM25_LEMMATIZE = True  # лемматизация корпуса и запросов для всех BM25-источников
# Приоритет слева направо; порядок используемых моделей берём из этого списка.
SCORE_ORDER = ["bm25_title", "giga", "bge", "bm25_description", "bm25_infm"]
SCORE_WEIGHTS = None  # для суммы: например {"bm25_title": 2}; отсутствующие веса равны 1

## Окружение и готовые индексы

Укажите DENSE_INDEX_DIR с папками giga/ и bge/ из эксперимента query–title.
Для локального запуска без dense задайте DENSE_MODELS = []. Индексы объявлений
повторно не строятся для dense; энкодеры нужны только для запросов.

In [ ]:
import sys
import subprocess

if INSTALL_DEPENDENCIES:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "bm25s>=0.3.11", "pymorphy3>=2", "razdel>=0.5",
        "polars>=1.44,<2", "pandas>=2", "scikit-learn>=1.6,<2",
        "faiss-cpu>=1.10,<2", "sentence-transformers>=5.1,<6", "transformers>=4.51,<5",
    ])

roots = [Path.cwd(), *Path.cwd().parents, *Path("/kaggle/working").glob("*")]
ROOT = Path(REPO_PATH) if REPO_PATH else next((p for p in roots if (p / "src/retrievers/geo_first.py").is_file()), None)
if ROOT is None or not (ROOT / "src/retrievers/geo_first.py").is_file():
    raise FileNotFoundError("Укажите REPO_PATH на обновлённый код с src/retrievers/geo_first.py.")
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import gc
import hashlib
import json
from time import perf_counter
import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display
from src.split import split_queries
from src.retrievers.bm25 import BM25Retriever
from src.retrievers.geo_first import GeoFirstRetriever

if DATA_PATH is None:
    matches = [p for p in [ROOT / "data/train.parquet", *Path("/kaggle/input").rglob("train.parquet")] if p.is_file()]
    if len(matches) != 1:
        raise ValueError(f"Укажите DATA_PATH: найдено {len(matches)} файлов train.parquet.")
    DATA_PATH = matches[0]
DATA_PATH = Path(DATA_PATH)
if DENSE_INDEX_DIR is None:
    DENSE_INDEX_DIR = Path("/kaggle/working/dense_indexes") if Path("/kaggle/working").is_dir() else ROOT / "dense_indexes"
DENSE_INDEX_DIR = Path(DENSE_INDEX_DIR)
ARTIFACTS = (Path("/kaggle/working") if Path("/kaggle/working").is_dir() else ROOT) / "artifacts/geo_first"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
assert 1 <= K <= 50 and all(n >= K for n in NS)
assert QUERY_BATCH_SIZE > 0

# Проверяем оба индекса до дорогой части эксперимента.
dense_metadata = {}
for name in DENSE_MODELS:
    path = DENSE_INDEX_DIR / name
    if not all((path / f).is_file() for f in ("index.faiss", "metadata.json")):
        raise FileNotFoundError(f"Нет сохранённого индекса {path}. Укажите DENSE_INDEX_DIR или DENSE_MODELS=[].")
    raw = (path / "metadata.json").read_bytes()
    metadata = json.loads(raw)
    params = metadata["params"]
    if (params["model"] != name or params["item_text_column"] != "item_title_raw"
            or params["query_text_column"] != "search_query"):
        raise ValueError(f"{path}: нужен индекс {name} для search_query → item_title_raw.")
    dense_metadata[name] = {"digest": hashlib.sha256(raw).hexdigest(), "params": params}
    del metadata, raw
print("Данные:", DATA_PATH, "\nИндексы:", DENSE_INDEX_DIR, "\nАртефакты:", ARTIFACTS)

## Тот же сплит и центры по train

Основной центр — медиана координат по item_location_id. Fallback — медиана по
объявлениям, связанным с search_location_id в train, после удаления повторов пары
(локация запроса, item_id). Число опорных объявлений сохраняется в center_items.
Координаты val/test-взаимодействий не используются для оценки центров.

Объявления без валидных координат не входят в геопул. Запрос без известного центра
получает пустую выдачу и Recall=0; такие запросы не исключаются из метрик.

In [ ]:
from src.geo import location_centers

QUERY_COLS = ["search_query", "search_location_id", "search_is_delivery_search", "search_infm_params_text", "search_category"]
GEO_COLS = ["item_location_id", "item_latitude", "item_longitude"]
data = pl.read_parquet(DATA_PATH, columns=QUERY_COLS + ["item_id"] + GEO_COLS)
data = data.with_columns(
    pl.Series("split", split_queries(data["search_query"], seed=SEED)),
    pl.col("item_latitude", "item_longitude").cast(pl.Float64),
)
items = pd.DataFrame(
    data.select("item_id", *GEO_COLS).unique(subset="item_id", keep="last", maintain_order=True)
    .sort("item_id").to_dict(as_series=False)
)
train_items = data.filter(pl.col("split") == "train").select("search_location_id", "item_id", *GEO_COLS)
centers = location_centers(pd.DataFrame(train_items.to_dict(as_series=False)))
queries, targets = {}, {}
for split in ("val", "test"):
    part = (data.filter(pl.col("split") == split).group_by(QUERY_COLS)
            .agg(pl.col("item_id").unique().alias("targets")).sort(QUERY_COLS))
    if N_QUERIES is not None:
        part = part.sample(n=min(N_QUERIES, part.height), seed=SEED, shuffle=True)
    queries[split] = pd.DataFrame(part.select(QUERY_COLS).to_dict(as_series=False)).join(centers, on="search_location_id")
    targets[split] = [set(ids) for ids in part["targets"].to_list()]
    assert all(targets[split])
    assert set().union(*targets[split]).issubset(set(items.item_id))
    print(split, "контекстов:", len(part), "без центра:", queries[split].center_lat.isna().sum())
display(data.group_by("split").len().sort("split"))
del data, train_items, part
gc.collect()

# Длинные тексты читаем только при построении соответствующего BM25-индекса.
def read_item_text(column):
    frame = (pl.scan_parquet(DATA_PATH).select("item_id", column)
             .unique(subset="item_id", keep="last", maintain_order=True).sort("item_id").collect())
    return pd.DataFrame(frame.to_dict(as_series=False))
for split, q in queries.items():
    print(split, "источник центра:")
    display(q.center_source.value_counts(dropna=False))

## Геопулы: ближайшие 5000, без текстового поиска

BallTree ищет ближайшие объявления по гаверсинусу. На границе пула одинаковое
расстояние разрешаем по item_id, чтобы top-50 был префиксом top-500 и top-5000.
Кеш зависит от данных, запросов, центров, кода, размера батча и максимального N.

In [ ]:
def save_frame(frame, path, schema):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".tmp.parquet")
    pl.DataFrame(frame.to_dict(orient="list"), schema=schema).write_parquet(temporary)
    temporary.replace(path)


def read_frame(path):
    return pd.DataFrame(pl.read_parquet(path).to_dict(as_series=False))


stat = DATA_PATH.stat()
query_hashes = {split: hashlib.sha256(pd.util.hash_pandas_object(q, index=True).to_numpy().tobytes()).hexdigest()
                for split, q in queries.items()}
pool_signature = {
    "data": [str(DATA_PATH.resolve()), stat.st_size, stat.st_mtime_ns], "queries": query_hashes,
    "max_n": max(NS), "batch": QUERY_BATCH_SIZE,
    "code": hashlib.sha256((ROOT / "src/retrievers/geo_first.py").read_bytes()).hexdigest(),
}
pool_digest = hashlib.sha256(json.dumps(pool_signature, sort_keys=True).encode()).hexdigest()[:16]
POOL_DIR = ARTIFACTS / "pools" / pool_digest
POOL_DIR.mkdir(parents=True, exist_ok=True)
(POOL_DIR / "manifest.json").write_text(json.dumps(pool_signature, ensure_ascii=False, indent=2))
jobs = [(split, start) for split, q in queries.items() for start in range(0, len(q), QUERY_BATCH_SIZE)]
geo = GeoFirstRetriever(n_candidates=max(NS)).fit(items)
for split, start in jobs:
    path = POOL_DIR / split / f"{start:07d}.parquet"
    if not path.is_file():
        pool = geo.candidates(queries[split].iloc[start:start + QUERY_BATCH_SIZE])
        save_frame(pool, path, {"query_index": pl.Int64, "item_id": pl.String,
                                "geo_rank": pl.Int64, "distance_km": pl.Float64})
del geo
print("Геопулы готовы:", POOL_DIR)

## Скоры всех моделей внутри геопула

Каждая модель последовательно скорит все пары из кеша. Нет вектора объявления
или пустой dense-запрос — NaN, а не искусственный нулевой cosine. BM25 без совпадений
даёт 0. Скоры сохраняются вместе с ключами пар; при чтении проверяем порядок.
BM25S суммирует postings общего индекса, затем забираем скоры нужных item_id.
Dense восстанавливает из FAISS только векторы кандидатов и считает скалярное произведение.

In [ ]:
sources = {
    "bm25_title": {"kind": "bm25", "item_text_column": "item_title_raw", "query_text_column": "search_query"},
    "bm25_description": {"kind": "bm25", "item_text_column": "item_description_raw", "query_text_column": "search_query"},
    "bm25_infm": {"kind": "bm25", "item_text_column": "item_infm_params_text", "query_text_column": "search_infm_params_text"},
}
for spec in sources.values():
    spec.update(lemmatize=BM25_LEMMATIZE, k1=1.5, b=0.75)
sources.update({name: {"kind": "dense", **dense_metadata[name]} for name in DENSE_MODELS})
assert set(sources).issubset(SCORE_ORDER) and len(SCORE_ORDER) == len(set(SCORE_ORDER))
cache_dirs = {}
for name, spec in sources.items():
    code_path = ROOT / "src/retrievers" / ("bm25.py" if spec["kind"] == "bm25" else "dense.py")
    signature = {"source": spec, "pool": pool_signature,
                 "code": hashlib.sha256(code_path.read_bytes()).hexdigest()}
    if spec["kind"] == "dense":
        index_stat = (DENSE_INDEX_DIR / name / "index.faiss").stat()
        signature["index"] = [index_stat.st_size, index_stat.st_mtime_ns]
    digest = hashlib.sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()[:16]
    cache_dirs[name] = ARTIFACTS / "scores" / name / digest
    cache_dirs[name].mkdir(parents=True, exist_ok=True)
    (cache_dirs[name] / "manifest.json").write_text(json.dumps(signature, ensure_ascii=False, indent=2))
    pending = [(split, start) for split, start in jobs if not (cache_dirs[name] / split / f"{start:07d}.parquet").is_file()]
    if not pending:
        print(name, "— все скоры в кеше")
        continue
    started = perf_counter()
    if spec["kind"] == "bm25":
        corpus = read_item_text(spec["item_text_column"])
        model = BM25Retriever(**{key: value for key, value in spec.items() if key != "kind"},
                              batch_size=QUERY_BATCH_SIZE).fit(corpus)
    else:
        from src.retrievers.dense import DenseRetriever
        corpus = read_item_text("item_title_raw")
        model = DenseRetriever.load(DENSE_INDEX_DIR / name, device=DEVICE)
        expected = corpus.loc[corpus.item_title_raw.fillna("").str.strip().ne(""), "item_id"].tolist()
        if model.item_ids_.tolist() != expected:
            raise ValueError(f"{name}: корпус не соответствует сохранённому индексу.")
        model.batch_size = DENSE_BATCH_SIZE
        del expected
    del corpus
    print(name, "— индекс готов за", round(perf_counter() - started, 1), "с")
    for i, (split, start) in enumerate(pending, 1):
        pool = read_frame(POOL_DIR / split / f"{start:07d}.parquet")
        result = pool[["query_index", "item_id"]].copy()
        result["score"] = model.score_candidates(queries[split].iloc[start:start + QUERY_BATCH_SIZE], pool)
        assert not np.isinf(result.score.to_numpy(dtype=float)).any()
        save_frame(result, cache_dirs[name] / split / f"{start:07d}.parquet",
                   {"query_index": pl.Int64, "item_id": pl.String, "score": pl.Float64})
        if i % 10 == 0 or i == len(pending):
            print(name, f"{i}/{len(pending)} батчей; {perf_counter() - started:.1f} с")
    del model, result
    gc.collect()
    if spec["kind"] == "dense":
        import torch
        torch.cuda.empty_cache()

## Сравнение кортежа и суммы

Проверяем BM25-title, три BM25 вместе и все пять моделей. В кортеж входят скоры
только выбранного набора, в порядке SCORE_ORDER. Дополнительный контроль — чистое
расстояние. При N=K=50 любое переранжирование полного пула сохраняет Recall@50;
различия стратегий по этой метрике ожидаются при N>K.

pool_recall — доля релевантных объявлений в геопуле; oracle_recall@50 — потолок
при идеальном порядке пула. Эти метрики одинаковы для всех моделей при одном N.
Если геопул отсекает релевантные объявления, текстовые модели вернуть их уже не смогут.

Обычное использование обёртки:
```python
model = GeoFirstRetriever({"bm25_title": fitted_bm25, "giga": loaded_giga},
                          n_candidates=500, fusion="tuple",
                          score_order=("bm25_title", "giga")).fit(items)
result = model.retrieve(queries_with_centers, k=50)
```

In [ ]:
source_sets = {"bm25_title": ["bm25_title"], "bm25_all": ["bm25_title", "bm25_description", "bm25_infm"], "geo": []}
if DENSE_MODELS:
    source_sets["all"] = list(sources)
configurations = pd.DataFrame([
    {"sources": group, "N": n, "fusion": fusion,
     "score_order": json.dumps([name for name in SCORE_ORDER if name in names]),
     "weights": json.dumps(SCORE_WEIGHTS)}
    for group, names in source_sets.items() for n in NS
    for fusion in (["distance"] if group == "geo" else ["tuple", "sum"])
])
CONFIG_COLS = ["sources", "N", "fusion", "score_order", "weights"]


def recall_per_query(result, relevant):
    found = result.groupby("query_index").item_id.agg(set).to_dict()
    return np.array([len(ids & found.get(i, set())) / len(ids) for i, ids in enumerate(relevant)])


def evaluate(split, configs):
    configs = configs.reset_index(drop=True)
    totals = np.zeros((len(configs), 4))
    needed = set().union(*(json.loads(order) for order in configs.score_order))
    q = queries[split]
    for start in range(0, len(q), QUERY_BATCH_SIZE):
        batch = q.iloc[start:start + QUERY_BATCH_SIZE]
        relevant = targets[split][start:start + len(batch)]
        denominators = np.array([len(ids) for ids in relevant])
        scored = read_frame(POOL_DIR / split / f"{start:07d}.parquet")
        for name in needed:
            part = read_frame(cache_dirs[name] / split / f"{start:07d}.parquet")
            assert np.array_equal(part[["query_index", "item_id"]].to_numpy(), scored[["query_index", "item_id"]].to_numpy())
            scored[name] = part.score.to_numpy()
        for n, subset in configs.groupby("N", sort=False):
            pool = scored.loc[scored.geo_rank <= n]
            hits = [item_id in relevant[i] for i, item_id in pool[["query_index", "item_id"]].itertuples(index=False, name=None)]
            positions = pool.query_index.to_numpy(dtype=int)
            found = np.bincount(positions, weights=hits, minlength=len(batch))
            size = np.bincount(positions, minlength=len(batch))
            coverage = (found / denominators).sum()
            oracle = (np.minimum(found, K) / denominators).sum()
            for config_i, row in subset.iterrows():
                model = GeoFirstRetriever(n_candidates=int(n), fusion=row.fusion,
                                          score_order=tuple(json.loads(row.score_order)), weights=json.loads(row.weights))
                result = model.rerank(pool, k=K)
                totals[config_i] += [recall_per_query(result, relevant).sum(), coverage, oracle, size.sum()]
        print(split, f"{min(start + len(batch), len(q))}/{len(q)} запросов", end="\r")
    print()
    result = configs.copy()
    result[[f"recall@{K}", "pool_recall", f"oracle_recall@{K}", "mean_pool_size"]] = totals / len(q)
    return result

## Validation: фиксируем лучший N и набор моделей

Выбираем по val лучшую конфигурацию каждой стратегии и одного общего победителя.
При равенстве Recall берём меньший N. Приоритет кортежа и веса также сохранены
в строках конфигураций, поэтому test использует выбранные настройки.

In [ ]:
validation = evaluate("val", configurations)
validation.to_csv(ARTIFACTS / "validation.csv", index=False)
display(validation.pivot_table(index=["sources", "N"], columns="fusion", values=f"recall@{K}"))
display(validation[["N", "pool_recall", f"oracle_recall@{K}", "mean_pool_size"]].drop_duplicates())
ordered = validation.sort_values([f"recall@{K}", "N", "sources", "fusion"], ascending=[False, True, True, True])
best = ordered.drop_duplicates("fusion").copy()
winner = ordered.iloc[0]
best.to_csv(ARTIFACTS / "selected_on_val.csv", index=False)
print("Выбран на val:", winner[CONFIG_COLS].to_dict())
display(best)

## Test: зафиксированные конфигурации

Считаем test для выбранных на val кортежа, суммы и чистого расстояния.
Общий победитель зафиксирован выше. Для сравнения с порядком «текст → гео» используйте
Hybrid_geo_pipeline с теми же данными, SEED и N_QUERIES: подготовка центров общая.

In [ ]:
comparison = evaluate("test", best[CONFIG_COLS]).merge(
    validation[CONFIG_COLS + [f"recall@{K}"]].rename(columns={f"recall@{K}": f"val_recall@{K}"}),
    on=CONFIG_COLS, validate="one_to_one",
).rename(columns={f"recall@{K}": f"test_recall@{K}"})
comparison["selected_on_val"] = comparison[CONFIG_COLS].eq(winner[CONFIG_COLS]).all(axis=1)
comparison.to_csv(ARTIFACTS / "test.csv", index=False)
display(comparison)
print("Результаты и кеш:", ARTIFACTS)